In [ ]:
import re
from typing import Annotated, List, Tuple
from typing_extensions import TypedDict

from kagraph import END, START, StateGraph, prompt_llm, add_messages, invoke_llm
from kagraph.llms import load_llm
from kagraph.messages import AnyMessage, HumanMessage, SystemMessage
from kagraph.prebuilt import ToolNode, tools_condition
from kagraph.tracing import trace

In [ ]:
def calculate(expression: str) -> str:
    """Evaluate arithmetic with numbers, spaces, parentheses, and + - * / only."""
    if not re.fullmatch(r"[0-9+\-*/().\s]+", expression):
        raise ValueError("unsupported expression characters")
    return str(eval(expression, {"__builtins__": {}}, {}))

def lookup_fact(query: str) -> str:
    """Look up a small local fact base for deterministic tutorial facts."""
    facts = {
        "france capital": "The capital of France is Paris.",
        "capital of france": "The capital of France is Paris.",
        "paris": "Paris is the capital city of France.",
    }
    normalized = query.lower()
    for key, value in facts.items():
        if key in normalized:
            return value
    raise ValueError(f"No local fact found for query: {query}")

def _parse_steps(text: str) -> list[str]:
    steps = []
    for line in text.splitlines():
        match = re.match(r"\s*\d+[\).]\s*(.+)", line)
        if match:
            steps.append(match.group(1).strip())
    if not steps:
        raise ValueError(f"Could not parse plan steps from:\n{text}")
    return steps

In [ ]:
class PlanExecuteInput(TypedDict):
    input: str

class PlanExecuteState(TypedDict, total=False):
    input: str
    plan: List[str]
    past_steps: Annotated[List[Tuple[str, str]], list.__add__]
    response: str
    
class ExecutorState(TypedDict, total=False):
    messages: Annotated[List[AnyMessage], add_messages]

class ExecutorInput(TypedDict):
    task: str

In [ ]:
PLANNER_PROMPT = """Create a short execution plan for this task.

Task: {task}

Use one step per line in this exact format:
1. <step>
2. <step>

The available executor tools are calculate(expression) and lookup_fact(query).
"""

REPLAN_PROMPT = """You are supervising a plan-and-execute agent.

Original task:
{task}

Original plan:
{plan}

Completed steps:
{past_steps}

If the task is fully solved, write FINAL: <answer>.
Otherwise write PLAN: followed by the remaining steps, one per numbered line.
"""

In [ ]:
def build_executor(llm, tools, prompt: str):
    def prepare_task(state: ExecutorInput):
        return {"messages": [HumanMessage(f"Execute this plan step: {state['task']}")]}

    def agent_node(state: ExecutorState):
        sys_msg = SystemMessage(prompt)
        response = invoke_llm(llm, prompt=None, messages=state["messages"], system=sys_msg.content, tools=tools)
        return {"messages": [response]}

    workflow = StateGraph(ExecutorState, input_schema=ExecutorInput)
    workflow.add_node("prepare_task", prepare_task, input_schema=ExecutorInput)
    workflow.add_node("agent", agent_node)
    workflow.add_node("tools", ToolNode(tools))
    workflow.add_edge(START, "prepare_task")
    workflow.add_edge("prepare_task", "agent")
    workflow.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
    workflow.add_edge("tools", "agent")
    return workflow.compile(name="executor")

In [ ]:
def build_graph(llm):
    executor = build_executor(
        llm,
        [calculate, lookup_fact],
        prompt=(
            "You are the executor in a plan-and-execute system. Use tools when they are relevant. "
            "For arithmetic, call calculate. For country-capital facts, call lookup_fact. "
            "Return a concise result for the assigned step."
        ),
    )
    graph = StateGraph(PlanExecuteState, input_schema=PlanExecuteInput)

    def plan_step(state: PlanExecuteState):
        plan_text = str(prompt_llm(llm, PLANNER_PROMPT.format(task=state["input"])))
        return {"plan": _parse_steps(plan_text)}

    def execute_step(state: PlanExecuteState):
        task = state["plan"][0]
        result = executor.invoke(
            {"task": task},
            chat_name="KaGraph Plan Executor tutorial",
            recursion_limit=15,
        )
        return {"past_steps": [(task, result["messages"][-1].content)]}

    execute_step.__kagraph_subgraph__ = executor

    def replan_step(state: PlanExecuteState):
        remaining_plan = state["plan"][1:]
        if remaining_plan:
            return {"plan": remaining_plan}

        replanner_output = str(prompt_llm(
            llm,
            REPLAN_PROMPT.format(
                task=state["input"],
                # Reconstruct the full plan by combining past executed steps and the current remaining step
                plan="\n".join(f"{idx + 1}. {step}" for idx, step in enumerate([s[0] for s in state["past_steps"]] + remaining_plan)),
                past_steps="\n".join(f"- {step}: {result}" for step, result in state["past_steps"]),
            ),
        ))
        if replanner_output.strip().upper().startswith("FINAL:"):
            return {"response": replanner_output.split(":", 1)[1].strip()}
        return {"plan": _parse_steps(replanner_output)}

    def should_end(state: PlanExecuteState):
        return END if state.get("response") else "execute"

    graph.add_node("planner", plan_step)
    graph.add_node("execute", execute_step)
    graph.add_node("replan", replan_step)
    graph.add_edge(START, "planner")
    graph.add_edge("planner", "execute")
    graph.add_edge("execute", "replan")
    graph.add_conditional_edges("replan", should_end, {END: END, "execute": "execute"})
    return graph.compile(name="plan_and_execute")

In [ ]:
llm = load_llm("qwen/qwen3-235b-a22b-instruct-2507", support_structured_outputs=True)
app = build_graph(llm)

In [ ]:
app.get_graph().draw_png()

In [ ]:
with trace("PlanAndExecute", include_agent_binary=True):
    result = app.invoke(
        {"input": "What is the capital of France, and what is (17 * 23) + 19?"},
        chat_name="KaGraph Plan and Execute tutorial",
        recursion_limit=20,
    )